In [ ]:
import fastai.learner as _fl
from fastai.learner import Recorder as _Recorder
from fastai.callback.tracker import SaveModelCallback as _SaveModelCallback

In [ ]:
_orig_load_model = _fl.load_model

def _dml_safe_load_model(file, model, opt, with_opt=True, device=None, strict=True, **kwargs):
    """
    DirectML-safe load_model replacement.

    Problems solved:
    1. DirectML can't handle map_location=<torch.device> — load to CPU first
    2. Optimizer state (grad_avg, sqr_avg) lands on CPU after load — move to DML
    3. Lookahead slow_weights must match their corresponding fast param device
       exactly — out_proj is intentionally on CPU, all others on DML
    """
    # Step 1: load everything to CPU
    _orig_load_model(file, model, opt, with_opt=with_opt,
                     device=torch.device('cpu'), strict=strict, **kwargs)

    # Step 2: resolve target device
    if device is not None:
        target_device = device
    else:
        param = next(
            (p for name, p in model.named_parameters() if 'out_proj' not in name),
            None
        )
        target_device = param.device if param is not None else torch.device('cpu')

    # Step 3: move model to DML
    model.to(target_device)

    # Step 4: re-pin out_proj to CPU — must happen before optimizer state move
    # so that param_lists reflects the correct final device for each parameter
    if hasattr(model, 'out_proj'):
        model.out_proj.proj.to(torch.device('cpu'))

    # Step 5: move inner Adam state to DML
    # opt is Lookahead wrapper — inner Adam is opt.opt
    if opt is not None:
        inner_opt = opt.opt if hasattr(opt, 'opt') else opt
        if hasattr(inner_opt, 'state'):
            for p, state in inner_opt.state.items():
                for k, v in state.items():
                    if isinstance(v, torch.Tensor):
                        state[k] = v.to(p.device)  # match param device, not target_device
                                                     # handles out_proj state staying on CPU

        # Step 6: move Lookahead slow_weights — match each to its fast param device
        # param_lists and slow_weights have identical structure: L(L(tensor))
        # out_proj fast param is on CPU → its slow weight must stay on CPU
        # all other fast params are on DML → their slow weights go to DML
        if hasattr(opt, 'slow_weights') and opt.slow_weights is not None:
            try:
                moved = L()
                for slow_pg, fast_pg in zip(opt.slow_weights, opt.param_lists):
                    moved_pg = L()
                    for slow_w, fast_p in zip(slow_pg, fast_pg):
                        moved_pg.append(slow_w.to(fast_p.device))
                    moved.append(moved_pg)
                opt.slow_weights = moved

                # Verify no mismatches remain
                mismatches = [
                    (slow_w.device, fast_p.device)
                    for slow_pg, fast_pg in zip(opt.slow_weights, opt.param_lists)
                    for slow_w, fast_p in zip(slow_pg, fast_pg)
                    if str(slow_w.device) != str(fast_p.device)
                ]
                if mismatches:
                    raise RuntimeError(f"slow_weights device mismatch: {mismatches}")

                print(f"[_dml_safe_load_model] slow_weights moved OK — "
                      f"{sum(len(pg) for pg in opt.slow_weights)} tensors, "
                      f"count={opt.count}, next_sync={opt.k - (opt.count % opt.k)}")

            except Exception as e:
                print(f"[_dml_safe_load_model] slow_weights move failed ({e}), resetting to None")
                opt.slow_weights = None
                opt.count = 0

_fl.load_model = _dml_safe_load_model

In [ ]:
def _patched_recorder_after_epoch(self):
    # Skipped epochs (CancelEpochException from SkipToEpoch) never run
    # after_train or after_validate, so log stays at [epoch_number] only.
    # Real epochs always have at minimum [epoch, train_loss, valid_loss]
    # regardless of epoch number including 0.
    min_expected = 3  # epoch + train_loss + valid_loss
    if len(self.log) < min_expected:
        return
    _orig_recorder_after_epoch(self)

In [ ]:
class SafeSaveModelCallback(_SaveModelCallback):
    """
    Guards against two resume failures in SaveModelCallback:
    
    1. IndexError in after_epoch: recorder.values[-1] exists but is shorter
       than self.idx — happens when start_epoch > 0 and first epoch hasn't
       fully populated recorder yet
       
    2. IndexError in after_fit: learn.load() inside after_fit hits same issue
       when loading best model at end of training
    """
    def after_epoch(self):
        vals = self.learn.recorder.values
        if not vals:
            return
        if len(vals[-1]) <= self.idx:
            return
        super().after_epoch()

    def after_fit(self, **kwargs):
        try:
            super().after_fit(**kwargs)
        except (IndexError, Exception) as e:
            print(f"[SafeSaveModelCallback] after_fit skipped: {e}")